# Actividad 1 - Herencia
**Estudiante:** 20211140003  

### Explicación
La herencia permite crear clases hijas a partir de una clase padre para reutilizar atributos y funciones comunes con `super()`.

- Clase padre: `EquipoRefrigeracion` (guarda `id_equipo`, `marca`, `potencia_hp`, `refrigerante`, `horas_operacion` y calcula consumo base).
- Subclase 1: `CompresorTornillo` (hereda de `EquipoRefrigeracion`, agrega presión de descarga y ajusta el consumo según su modulación).
- Subclase 2: `EvaporadorForzado` (hereda de `EquipoRefrigeracion`, agrega número de ventiladores y control de deshielo).

### Diagrama UML
```plantuml
@startuml
skinparam classAttributeIconSize 0

class EquipoRefrigeracion {
    # id_equipo: str
    # marca: str
    # potencia_hp: float
    # refrigerante: str
    # horas_operacion: float
    --
    + __init__(id_equipo: str, marca: str, potencia_hp: float, refrigerante: str, horas_iniciales: float)
    + registrar_horas(horas: float): None
    + estimar_consumo_electrico_kwh(horas_calculo: float): float
    + __str__(): str
}

class CompresorTornillo {
    - _presion_descarga_psi: float
    - _capacidad_modulacion_pct: float
    --
    + __init__(id_equipo: str, marca: str, potencia_hp: float, refrigerante: str, presion_descarga: float)
    + ajustar_modulacion(nuevo_porcentaje: float): bool
    + estimar_consumo_electrico_kwh(horas_calculo: float): float
    + __str__(): str
}

class EvaporadorForzado {
    - _numero_ventiladores: int
    - _modo_deshielo_activo: bool
    - _tiempo_deshielo_minutos: int
    --
    + __init__(id_equipo: str, marca: str, potencia_hp: float, refrigerante: str, num_ventiladores: int, tiempo_deshielo: int)
    + iniciar_deshielo(): str
    + finalizar_deshielo(): str
    + __str__(): str
}

EquipoRefrigeracion <|-- CompresorTornillo
EquipoRefrigeracion <|-- EvaporadorForzado
@enduml

```

In [1]:
# Clase base
class EquipoRefrigeracion:
    def __init__(self, id_equipo: str, marca: str, potencia_hp: float, refrigerante: str, horas_iniciales: float = 0.0) -> None:
        self.id_equipo: str = id_equipo
        self.marca: str = marca
        self.potencia_hp: float = potencia_hp
        self.refrigerante: str = refrigerante
        self.horas_operacion: float = max(0.0, horas_iniciales)

    def registrar_horas(self, horas: float) -> None:
        if horas > 0:
            self.horas_operacion += horas
            print(f"[{self.id_equipo}] Sumadas {horas:.1f}h. Total acumulado: {self.horas_operacion:.1f}h")

    def estimar_consumo_electrico_kwh(self, horas_calculo: float) -> float:
        # 1 HP equivale a unos 0.7457 kW
        kw: float = self.potencia_hp * 0.7457
        return kw * horas_calculo

    def __str__(self) -> str:
        return f"{self.id_equipo} ({self.marca}) | {self.potencia_hp} HP | {self.refrigerante} | {self.horas_operacion:.1f}h"


# Subclase 1: Compresor
class CompresorTornillo(EquipoRefrigeracion):
    def __init__(self, id_equipo: str, marca: str, potencia_hp: float, refrigerante: str, presion_descarga: float = 180.0) -> None:
        super().__init__(id_equipo, marca, potencia_hp, refrigerante, horas_iniciales=0.0)
        self._presion_descarga_psi: float = presion_descarga
        self._capacidad_modulacion_pct: float = 100.0

    def ajustar_modulacion(self, nuevo_porcentaje: float) -> bool:
        if 20.0 <= nuevo_porcentaje <= 100.0:
            self._capacidad_modulacion_pct = nuevo_porcentaje
            print(f"[{self.id_equipo}] Modulacion ajustada a {self._capacidad_modulacion_pct:.0f}%")
            return True
        print(f"[{self.id_equipo}] Porcentaje inválido ({nuevo_porcentaje}%)")
        return False

    def estimar_consumo_electrico_kwh(self, horas_calculo: float) -> float:
        # Usa el calculo de la clase padre y lo ajusta por la modulacion
        consumo_base: float = super().estimar_consumo_electrico_kwh(horas_calculo)
        return consumo_base * (self._capacidad_modulacion_pct / 100.0)

    def __str__(self) -> str:
        return f"[Compresor] {super().__str__()} | Presion: {self._presion_descarga_psi} PSI | Carga: {self._capacidad_modulacion_pct:.0f}%"


# Subclase 2: Evaporador
class EvaporadorForzado(EquipoRefrigeracion):
    def __init__(self, id_equipo: str, marca: str, potencia_hp: float, refrigerante: str, num_ventiladores: int, tiempo_deshielo: int = 30) -> None:
        super().__init__(id_equipo, marca, potencia_hp, refrigerante, horas_iniciales=0.0)
        self._numero_ventiladores: int = num_ventiladores
        self._modo_deshielo_activo: bool = False
        self._tiempo_deshielo_minutos: int = tiempo_deshielo

    def iniciar_deshielo(self) -> str:
        self._modo_deshielo_activo = True
        return f"[{self.id_equipo}] Deshielo prendido por {self._tiempo_deshielo_minutos} min (ventiladores apagados)"

    def finalizar_deshielo(self) -> str:
        self._modo_deshielo_activo = False
        return f"[{self.id_equipo}] Deshielo apagado (se reactivan {self._numero_ventiladores} ventiladores)"

    def __str__(self) -> str:
        estado = "Deshielo ACTIVO" if self._modo_deshielo_activo else "Normal"
        return f"[Evaporador] {super().__str__()} | Ventiladores: {self._numero_ventiladores} | {estado}"

print("Clases de herencia definidas correctamente.")


Clases de herencia definidas correctamente.


In [2]:
# Creamos instancias de las subclases
compresor = CompresorTornillo("COMP-01", "Bitzer", potencia_hp=40.0, refrigerante="R-404A", presion_descarga=185.0)
evaporador = EvaporadorForzado("EVAP-01", "Bohn", potencia_hp=4.0, refrigerante="R-404A", num_ventiladores=3)

print("--- Objetos creados ---")
print(compresor)
print(evaporador)

print("\n--- Metodo heredado del padre (registrar_horas) ---")
compresor.registrar_horas(80.0)
evaporador.registrar_horas(80.0)

print("\n--- Consumo electrico con modulacion en compresor ---")
print(f"Consumo 10h al 100%: {compresor.estimar_consumo_electrico_kwh(10):.2f} kWh")
compresor.ajustar_modulacion(50.0)
print(f"Consumo 10h al 50%:  {compresor.estimar_consumo_electrico_kwh(10):.2f} kWh")

print("\n--- Control de deshielo en evaporador ---")
print(evaporador.iniciar_deshielo())
print(evaporador)
print(evaporador.finalizar_deshielo())
print(evaporador)


--- Objetos creados ---
[Compresor] COMP-01 (Bitzer) | 40.0 HP | R-404A | 0.0h | Presion: 185.0 PSI | Carga: 100%
[Evaporador] EVAP-01 (Bohn) | 4.0 HP | R-404A | 0.0h | Ventiladores: 3 | Normal

--- Metodo heredado del padre (registrar_horas) ---
[COMP-01] Sumadas 80.0h. Total acumulado: 80.0h
[EVAP-01] Sumadas 80.0h. Total acumulado: 80.0h

--- Consumo electrico con modulacion en compresor ---
Consumo 10h al 100%: 298.28 kWh
[COMP-01] Modulacion ajustada a 50%
Consumo 10h al 50%:  149.14 kWh

--- Control de deshielo en evaporador ---
[EVAP-01] Deshielo prendido por 30 min (ventiladores apagados)
[Evaporador] EVAP-01 (Bohn) | 4.0 HP | R-404A | 80.0h | Ventiladores: 3 | Deshielo ACTIVO
[EVAP-01] Deshielo apagado (se reactivan 3 ventiladores)
[Evaporador] EVAP-01 (Bohn) | 4.0 HP | R-404A | 80.0h | Ventiladores: 3 | Normal
